# Test ModelHub LLM Gateway (pakai LangChain)

1. `gemma-4-26B-A4B-it`
2. `telkom-ai-instruct`
3. `telkom-ai-vision-instruct`
4. `llm_mini`
5. `telkom-document-extraction-multimodal`
6. `qwen3-embedding-4b`
7. `telkom-ai-coder-instruct`

In [1]:
# %pip install -q langchain langchain-openai openai python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("MODELHUB_LLM_API_KEY")
BASE_URL = os.getenv("MODELHUB_LLM_URL", "https://api-modelhub.airplayground.id").rstrip("/")

if not BASE_URL.endswith("/v1"):
    BASE_URL = f"{BASE_URL}/v1"

if not API_KEY:
    raise RuntimeError("MODELHUB_LLM_API_KEY belum diset di .env")

print("Base URL :", BASE_URL)
print("API key ada?", bool(API_KEY))

Base URL : https://api-modelhub.aiplayground.id/v1
API key ada? True


In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

models = client.models.list()

print("Base URL:", BASE_URL)
print("\nModel yang tersedia:")

for model in models.data:
    print("-", model.id)

Base URL: https://api-modelhub.aiplayground.id/v1

Model yang tersedia:
- Qwen3-Embedding-0.6B
- watsonx-qwen3-30b-a3b-instruct-2507
- telkom-ai-vision-instruct
- Neutra-Qwen/Qwen3.6-35B-A3B
- llama-nemotron-rerank-1b-v2
- telkom-ai-coder-instruct
- llm_mini
- telkom-ai-instruct
- watsonx-qwen3-vl-8b-instruct
- Qwen3-Embedding-8B
- PaddleOCR-VL
- telkom-document-extraction-multimodal
- gemma-4-26B-A4B-it
- kimi-code
- s0/gpt/terra
- s0/gpt/luna
- qwen3.8-27b


In [ ]:
import time, uuid
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

MODELS = [
    ("watsonx-qwen3-30b-a3b-instruct-2507", "watsonx-qwen3-30b",        {}),
    ("gemma-4-26B-A4B-it",                  "gemma-4-26B",              {}),
    ("telkom-ai-instruct",                  "telkom-ai-instruct",       {}),
    ("llm_mini",                            "llm_mini",                 {}),
    ("qwen3.8-27b",                         "qwen3.8-27b [think:off]",  {"thinking": False}),
    ("qwen3.8-27b",                         "qwen3.8-27b [think:on]",   {"thinking": True, "max_tokens": 1500}),
    ("telkom-ai-coder-instruct",            "telkom-ai-coder-instruct", {}),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

SYSTEM_PROMPT = "Kamu asisten yang ringkas. Jawab dalam Bahasa Indonesia, maksimal 2 kalimat."
USER_PROMPT = "Sebutkan 3 manfaat RAG untuk chatbot perusahaan."

def smoke_test(model_name, label=None, config=None, temperature=0.3, max_tokens=256, timeout=120):
    config = config or {}
    label = label or model_name
    max_tokens = config.get("max_tokens", max_tokens)

    kwargs = dict(
        api_key=API_KEY,
        base_url=BASE_URL,
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        timeout=timeout,
        max_retries=0,              # biar error-nya kelihatan apa adanya
        default_headers=HEADERS,
    )
    if config.get("thinking") is False:
        kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}

    # penanda unik biar tiap panggilan selalu miss cache
    user_msg = USER_PROMPT + f"\n\n<!-- run:{uuid.uuid4().hex[:8]} -->"

    t0 = time.perf_counter()
    try:
        chat = ChatOpenAI(**kwargs)
        resp = chat.invoke([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=user_msg),
        ])
        elapsed = time.perf_counter() - t0

        usage = (resp.usage_metadata or {}) if hasattr(resp, "usage_metadata") else {}
        text = (resp.content or "").strip()

        status = "OK" if text else "EMPTY"
        if not text:
            ak = resp.additional_kwargs or {}
            reasoning = (ak.get("reasoning_content")
                         or (ak.get("provider_specific_fields") or {}).get("reasoning_content")
                         or "").strip()
            if reasoning:
                status = "THINKING"
                text = reasoning

        out_tok = usage.get("output_tokens")
        tok_s = round(out_tok / elapsed, 1) if out_tok and elapsed else None

        return {
            "model": label,
            "status": status,
            "latency_s": round(elapsed, 2),
            "in_tok": usage.get("input_tokens"),
            "out_tok": out_tok,
            "tok_per_s": tok_s,
            "preview": text[:160].replace("\n", " "),
            "error": "",
        }

    except Exception as e:
        return {
            "model": label,
            "status": "FAIL",
            "latency_s": round(time.perf_counter() - t0, 2),
            "in_tok": None,
            "out_tok": None,
            "tok_per_s": None,
            "preview": "",
            "error": f"{type(e).__name__}: {str(e)[:200]}",
        }


results = []
for i, (name, label, cfg) in enumerate(MODELS, 1):
    print(f"[{i}/{len(MODELS)}] testing {label} ...", end=" ", flush=True)
    r = smoke_test(name, label, cfg)
    results.append(r)
    print(f"{r['status']} ({r['latency_s']}s)")

print("\n" + "=" * 90)
print("HASIL SMOKE TEST")
print("=" * 90)
for r in results:
    print(f"\n[{r['status']}] {r['model']}  |  {r['latency_s']}s  |  in={r['in_tok']} out={r['out_tok']} | {r['tok_per_s']} tok/s")
    if r["error"]:
        print(f"   ERROR   : {r['error']}")
    else:
        print(f"   PREVIEW : {r['preview']}")

ok = sum(1 for r in results if r["status"] == "OK")
print(f"\n>>> {ok}/{len(MODELS)} konfigurasi berhasil dipanggil.")

# opsional, kalau pandas tersedia
try:
    import pandas as pd
    df = pd.DataFrame(results)[["model", "status", "latency_s", "in_tok", "out_tok", "tok_per_s", "error"]]
    display(df.sort_values(["status", "latency_s"]))
except ImportError:
    pass

[1/7] testing watsonx-qwen3-30b ... OK (3.89s)
[2/7] testing gemma-4-26B ... OK (2.0s)
[3/7] testing telkom-ai-instruct ... OK (2.14s)
[4/7] testing llm_mini ... OK (1.22s)
[5/7] testing qwen3.8-27b [think:off] ... OK (1.41s)
[6/7] testing qwen3.8-27b [think:on] ... OK (22.06s)
[7/7] testing telkom-ai-coder-instruct ... OK (1.25s)

HASIL SMOKE TEST

[OK] watsonx-qwen3-30b  |  3.89s  |  in=63 out=71 | 18.2 tok/s
   PREVIEW : RAG (Retrieval-Augmented Generation) memungkinkan chatbot perusahaan memberikan jawaban yang lebih akurat dengan mengacu pada dokumen internal terkini. Selain i

[OK] gemma-4-26B  |  2.0s  |  in=62 out=51 | 25.5 tok/s
   PREVIEW : Tiga manfaat RAG adalah memberikan jawaban yang lebih akurat berdasarkan data internal perusahaan, mengurangi risiko halusinasi AI, dan memudahkan pembaruan inf

[OK] telkom-ai-instruct  |  2.14s  |  in=63 out=71 | 33.1 tok/s
   PREVIEW : RAG (Retrieval-Augmented Generation) memungkinkan chatbot perusahaan memberikan jawaban yang lebih aku

,model,status,latency_s,in_tok,out_tok,tok_per_s,error
3,llm_mini,OK,1.22,58,58,47.6,
6,telkom-ai-coder-instruct,OK,1.25,62,67,53.7,
4,qwen3.8-27b [think:off],OK,1.41,60,41,29.1,
1,gemma-4-26B,OK,2.00,62,51,25.5,
2,telkom-ai-instruct,OK,2.14,63,71,33.1,
0,watsonx-qwen3-30b,OK,3.89,63,71,18.2,
5,qwen3.8-27b [think:on],OK,22.06,92,1089,49.4,


In [11]:
# apakah model bisa browsing / pakai tool? ===
import json
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

PROBE_MODELS = [
    "watsonx-qwen3-30b-a3b-instruct-2507",
    "gemma-4-26B-A4B-it",
    "llm_mini",
    "telkom-ai-coder-instruct",
    "qwen3.8-27b",
]

# TEST 1: native web search (coba beberapa format proxy)
print("=" * 80)
print("TEST 1 — NATIVE WEB SEARCH")
print("=" * 80)

SEARCH_PAYLOADS = {
    "tools:web_search":          {"tools": [{"type": "web_search"}]},
    "tools:web_search_preview":  {"tools": [{"type": "web_search_preview"}]},
    "extra:web_search_options":  {"extra_body": {"web_search_options": {}}},
    "extra:enable_search":       {"extra_body": {"enable_search": True}},
}

m = PROBE_MODELS[0]
for label, payload in SEARCH_PAYLOADS.items():
    try:
        r = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": "Cari di web: siapa Menteri Perindustrian RI saat ini? Sebutkan sumbernya."}],
            max_tokens=200,
            **payload,
        )
        msg = r.choices[0].message
        print(f"\n[DITERIMA] {label}")
        print("  content   :", repr(msg.content)[:200])
        print("  tool_calls:", msg.tool_calls)
        extra = getattr(msg, "annotations", None) or getattr(msg, "provider_specific_fields", None)
        print("  annotations/citations:", str(extra)[:200])
    except Exception as e:
        print(f"[DITOLAK ] {label:28s} -> {type(e).__name__}: {str(e)[:110]}")

# TEST 2: function calling (fallback paling realistis)
print("\n" + "=" * 80)
print("TEST 2 — FUNCTION CALLING")
print("=" * 80)

TOOLS = [{
    "type": "function",
    "function": {
        "name": "cari_publikasi_resmi",
        "description": "Cari publikasi resmi pemerintah Indonesia di web.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Kata kunci pencarian"},
                "situs": {"type": "string", "description": "Domain, mis. kemenperin.go.id"},
            },
            "required": ["query"],
        },
    },
}]

for m in PROBE_MODELS:
    try:
        r = client.chat.completions.create(
            model=m,
            messages=[{
                "role": "user",
                "content": "Carikan laporan kinerja 2025 Kementerian Perindustrian. Gunakan tool yang tersedia.",
            }],
            tools=TOOLS,
            tool_choice="auto",
            max_tokens=300,
        )
        msg = r.choices[0].message
        tc = msg.tool_calls
        if tc:
            call = tc[0]
            print(f"[TOOL OK ] {m}")
            print(f"           -> {call.function.name}({call.function.arguments[:120]})")
        else:
            print(f"[NO CALL ] {m} | content: {repr(msg.content)[:90]}")
    except Exception as e:
        print(f"[GAGAL   ] {m} -> {type(e).__name__}: {str(e)[:110]}")

# TEST 3: knowledge cutoff (tanpa tool)
print("\n" + "=" * 80)
print("TEST 3 — PENGETAHUAN 2025 TANPA SEARCH")
print("=" * 80)

CUTOFF_Q = ("Apa isi Perpres Nomor 12 Tahun 2025 tentang RPJMN 2025-2029, "
            "dan berapa target indikator 'Produksi vanili' untuk tahun 2025? "
            "Jika tidak tahu, jawab 'TIDAK TAHU'.")

for m in PROBE_MODELS:
    try:
        r = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": CUTOFF_Q}],
            max_tokens=250,
            temperature=0.0,
        )
        txt = (r.choices[0].message.content or "").strip()
        if not txt:
            txt = "(kosong / thinking)"
        jujur = "TIDAK TAHU" in txt.upper()
        print(f"\n[{'JUJUR' if jujur else 'NGARANG?'}] {m}")
        print("  ", txt[:260].replace("\n", " "))
    except Exception as e:
        print(f"[GAGAL] {m} -> {type(e).__name__}: {str(e)[:100]}")

TEST 1 — NATIVE WEB SEARCH
[DITOLAK ] tools:web_search             -> BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'openai' returned HTTP 400.", 'type': None, 'param'
[DITOLAK ] tools:web_search_preview     -> BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'openai' returned HTTP 400.", 'type': None, 'param'

[DITERIMA] extra:web_search_options
  content   : 'Sebagai informasi terkini (hingga Juni 2024), Menteri Perindustrian Republik Indonesia saat ini adalah **Agus Gumiwang Kartasasmita**.\n\nBeliau menjabat sebagai Menteri Perindustrian sejak 20 Oktobe
  tool_calls: None
  annotations/citations: {'refusal': None}

[DITERIMA] extra:enable_search
  content   : 'Sebagai informasi terkini (per 5 April 2024), Menteri Perindustrian Republik Indonesia saat ini adalah **Agus Gumiwang Kartasasmita**.\n\nBeliau menjabat sejak 20 Oktober 2020, dalam Kabinet Indonesi
  tool_calls: None
  annotations/citations: {'refusal': None}

TEST 2 

In [4]:
ALL_CHAT = [
    "watsonx-qwen3-30b-a3b-instruct-2507",
    "telkom-ai-instruct",
    "telkom-ai-coder-instruct",
    "telkom-ai-vision-instruct",
    "watsonx-qwen3-vl-8b-instruct",
    "gemma-4-26B-A4B-it",
    "llm_mini",
    "qwen3.8-27b",
    "s0/gpt/terra",
    "s0/gpt/luna",
    "kimi-code",
]

PAYLOADS = {
    "baseline (tanpa apa2)":   {},
    "tools:web_search":        {"tools": [{"type": "web_search"}]},
    "tools:web_search_preview":{"tools": [{"type": "web_search_preview"}]},
    "extra:web_search_options":{"extra_body": {"web_search_options": {}}},
}

# pertanyaan yang MUSTAHIL dijawab tanpa search
Q = ("Berapa harga penutupan saham BBCA di BEI pada perdagangan terakhir, "
     "dan kapan tanggalnya? Sertakan URL sumbernya. Jika tidak tahu, jawab TIDAK TAHU.")

for m in ALL_CHAT:
    print(f"\n{'='*70}\n{m}")
    for label, payload in PAYLOADS.items():
        try:
            r = client.chat.completions.create(
                model=m,
                messages=[{"role": "user", "content": Q}],
                max_tokens=200, temperature=0.0, **payload,
            )
            msg = r.choices[0].message
            txt = (msg.content or "").strip() or "(kosong)"
            tahu = "TIDAK TAHU" not in txt.upper()
            ada_url = "http" in txt
            tanda = "SEARCH?" if (tahu and ada_url) else "no-search"
            print(f"  [{tanda:9s}] {label:26s} | {txt[:90]}")
        except Exception as e:
            print(f"  [ERROR    ] {label:26s} | {type(e).__name__}: {str(e)[:70]}")


watsonx-qwen3-30b-a3b-instruct-2507
  [no-search] baseline (tanpa apa2)      | TIDAK TAHU

Saya tidak dapat menyediakan data harga penutupan saham BBCA di Bursa Efek Ind
  [ERROR    ] tools:web_search           | BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'openai' re
  [ERROR    ] tools:web_search_preview   | BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'openai' re
  [no-search] extra:web_search_options   | TIDAK TAHU

Saya tidak dapat menyediakan data harga penutupan saham BBCA di Bursa Efek Ind

telkom-ai-instruct
  [no-search] baseline (tanpa apa2)      | TIDAK TAHU

Saya tidak dapat menyediakan data harga penutupan saham BBCA di Bursa Efek Ind
  [ERROR    ] tools:web_search           | BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'hosted_vll
  [ERROR    ] tools:web_search_preview   | BadRequestError: Error code: 400 - {'error': {'message': "Upstream provider 'hosted_vll
  [no-search] ext

## 1. gemma-4-26B-A4B-it

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="gemma-4-26B-A4B-it",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

content="ModelHub is amazing because it provides a seamless, centralized gateway to explore, test, and deploy the world's most cutting-edge AI models with incredible ease." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 50, 'total_tokens': 84, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma-4-26B-A4B-it', 'system_fingerprint': 'vllm-0.22.1-tp4-1ea8c02e', 'id': 'chatcmpl-3a0ea9b10277b57e208dcf299ae19e21', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a095ab-cbce-7742-8832-f4c92e2631f9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 34, 'total_tokens': 84, 'input_token_details': {}, 'output_token_details': {}}


## 2. telkom-ai-instruct

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="telkom-ai-instruct",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

content="It's amazing because it effortlessly understands and generates human-like text, making complex tasks feel simple and intuitive." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 43, 'total_tokens': 65, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'telkom-ai-instruct', 'system_fingerprint': None, 'id': 'chatcmpl-bf5f910d1e7a8dda8f26cb4086d5ea46', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a05714-9e7e-7003-9b4e-752a8d1a8659-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 22, 'total_tokens': 65, 'input_token_details': {}, 'output_token_details': {}}


## 3. telkom-ai-vision-instruct

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="telkom-ai-vision-instruct",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

content='ModelHub is amazing because it democratizes access to cutting-edge AI models, empowering developers and researchers to innovate faster with just a few lines of code.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 43, 'total_tokens': 74, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'telkom-ai-vision-instruct', 'system_fingerprint': 'vllm-0.22.1-f1f5c426', 'id': 'chatcmpl-a034e1160ff9b91e550301a4c8b7b7c8', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a05716-acfe-71c3-9ddc-5f73167eefaf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 31, 'total_tokens': 74, 'input_token_details': {}, 'output_token_details': {}}


## 4. llm_mini

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="llm_mini",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

content='ModelHub is amazing because it provides a centralized, accessible platform for discovering, managing, and deploying a wide variety of powerful AI models.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 46, 'total_tokens': 74, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'llm_mini', 'system_fingerprint': 'vllm-0.22.1-3eec686c', 'id': 'chatcmpl-7743e42e493b87b99a3bda8eb48a6283', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a05716-da00-7ff3-a430-c1e9d3191b15-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens': 28, 'total_tokens': 74, 'input_token_details': {}, 'output_token_details': {}}


## 5. telkom-document-extraction-multimodal

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="telkom-document-extraction-multimodal",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

InternalServerError: Error code: 524 - {'type': 'https://developers.cloudflare.com/support/troubleshooting/http-status-codes/cloudflare-5xx-errors/error-524/', 'title': 'Error 524: A timeout occurred', 'status': 524, 'detail': 'The origin web server did not return a complete response within the 120-second Proxy Read Timeout window. The connection was established, but the origin took too long to respond.', 'instance': 'a33aef5f9baae60e', 'error_code': 524, 'error_name': 'origin_response_timeout', 'error_category': 'origin', 'ray_id': 'a33aef5f9baae60e', 'timestamp': '2026-08-31T09:18:17Z', 'zone': 'api-modelhub.aiplayground.id', 'cloudflare_error': True, 'retryable': True, 'retry_after': 120, 'owner_action_required': True, 'what_you_should_do': '**Wait and retry.** Back off for at least 120 seconds. If the error persists, the website operator should check for long-running processes or an overloaded origin.', 'footer': 'This error was generated by Cloudflare on behalf of the website owne

## 6. Qwen3-Embedding-8B

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    openai_api_base=BASE_URL,
    openai_api_key=API_KEY,
    model="Qwen3-Embedding-8B",
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    },
)

vector = embeddings.embed_query("test from modelhub. tell me why it's amazing in 1 sentence")
print("panjang vektor:", len(vector))
print("5 nilai pertama:", vector[:5])

panjang vektor: 4096
5 nilai pertama: [0.0066545321606099606, -0.013541537337005138, -0.002731554675847292, -0.0093570277094841, 0.01592438295483589]


## 7. telkom-ai-coder-instruct

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOpenAI(
    api_key=os.getenv("MODELHUB_LLM_API_KEY"),
    openai_api_base=os.getenv("MODELHUB_LLM_URL"),
    model="telkom-ai-coder-instruct",
    temperature=float(os.getenv("MODELHUB_LLM_TEMPERATURE", 0.5)),
    max_tokens=int(os.getenv("MODELHUB_LLM_MAX_TOKENS", 1000)),
    default_headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
)

messages = [
    SystemMessage(content="You are a helpful assistant that im using to make a test request to."),
    HumanMessage(content="test from modelhub. tell me why it's amazing in 1 sentence"),
]

response = chat.invoke(messages)
print(response)

content="I notice you're testing from ModelHub, but I don't have any information about what specific model or service you're referring to. Could you clarify which model or feature you'd like me to comment on? For example, are you testing a language model, computer vision model, or something else? Once you provide more details, I'll be happy to explain why it's amazing in one sentence!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 43, 'total_tokens': 124, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'telkom-ai-coder-instruct', 'system_fingerprint': None, 'id': 'chatcmpl-f20fb004b1e5194a1a7a8b769725a09d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0572a-48be-7032-a971-97f69db7b62f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 43, 'output_tokens': 81, 'total_tokens': 124, 'input_token_details': {}, 'output_token_details